# Leaky AIC — Dilution Effect

**Figures generated:** aic_leaky.svg, aic_leaky_vs_eta.svgRun all cells below to regenerate.

In [ ]:
%matplotlib inline

In [ ]:
import numpy as npimport subprocess, timeimport matplotlib, matplotlib.pyplot as pltmatplotlib.rcParams.update({    "svg.fonttype": "path", "mathtext.fontset": "cm",    "font.family": "serif", "font.size": 14,    "axes.labelsize": 16, "axes.titlesize": 15,    "legend.fontsize": 11, "xtick.labelsize": 13, "ytick.labelsize": 13,    "lines.linewidth": 2.5, "axes.linewidth": 0.8,    "axes.spines.top": False, "axes.spines.right": False,})BLUE="#0072BD"; ORANGE="#D95319"; GREEN="#77AC30"; PURPLE="#7E2F8E"OUT = "/mnt/user-data/outputs"k = 1.0gc = 0.5  # fixed dilution raten_eta = 50nev = 3_000_000burn = 500_000eta_vals = np.logspace(-1, 3, n_eta)inp = " ".join(f"{v:.8f}" for v in eta_vals) + "\n"print(f"k={k}, gc={gc}, {n_eta} eta values, {nev/1e6:.0f}M events each")t0 = time.time()# Reuse the leaky SSA binary, but need to pass gc for each eta# The binary takes (k, eta, ngc, nev, burn) and sweeps gc.# I need the opposite: fixed gc, sweep eta. Let me call it once per eta.# Actually faster: just call it in a loop via subprocess for each eta,# or write a small wrapper.# Simpler: call the binary once per eta with ngc=1results = []for i, eta in enumerate(eta_vals):    proc = subprocess.run(        ["/home/claude/ssa_leaky", str(k), str(eta),         "1", str(nev), str(burn)],        input=f"{gc:.8f}\n", capture_output=True, text=True, timeout=30)    vals = [float(x) for x in proc.stdout.strip().split()]    results.append(vals)    if (i+1) % 10 == 0:        print(f"  {i+1}/{n_eta} done", flush=True)elapsed = time.time() - t0print(f"SSA done in {elapsed:.1f}s")mean_p = np.array([r[0] for r in results])mean_z1 = np.array([r[1] for r in results])mean_z2 = np.array([r[2] for r in results])# ── Figure ────────────────────────────────────────────────────fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)fig.subplots_adjust(left=0.11, right=0.96, top=0.92, bottom=0.10,                    hspace=0.18)fig.suptitle(    rf"Leaky AIC ($\gamma_c = {gc}$): increasing $\eta$ "    r"recovers adaptation  ($k\!=\!1$)",    fontsize=14, fontweight="bold")# Top: E[P] vs etaax1.semilogx(eta_vals, mean_p, "o-", color=BLUE, ms=4, lw=2.0,             label=r"$E[P]$ (SSA)")ax1.axhline(1.0, color=GREEN, lw=2.0, ls="--",            label=r"Set-point $\mu/\theta = 1$")ax1.fill_between(eta_vals, 1.0, mean_p, alpha=0.15, color=ORANGE)ax1.set_ylabel(r"$E[P]$")ax1.set_title("Mean protein at stationarity", fontsize=13)ax1.legend(loc="lower right", framealpha=0.95, edgecolor="0.75")ax1.grid(True, alpha=0.15)# Bottom: relative errorrel_err = np.abs(mean_p - 1.0) / 1.0 * 100ax2.loglog(eta_vals, rel_err, "o-", color=ORANGE, ms=4, lw=2.0,           label=r"$|E[P] - p^*|/p^*$ (%)")ax2.set_ylabel("Relative SS error (%)")ax2.set_xlabel(r"Sequestration rate $\eta$")ax2.set_title(    r"Error decreases as $\eta$ dominates over $\gamma_c$",    fontsize=13)ax2.legend(loc="upper right", framealpha=0.95, edgecolor="0.75")ax2.grid(True, alpha=0.15)fig.savefig("aic_leaky_vs_eta.svg", bbox_inches="tight")# fig.savefig("aic_leaky_vs_eta.png")  # uncomment to saveplt.close()print("Saved aic_leaky_vs_eta.svg")print(f"E[P] at eta=0.1: {mean_p[0]:.4f} (error {rel_err[0]:.1f}%)")print(f"E[P] at eta=1000: {mean_p[-1]:.4f} (error {rel_err[-1]:.1f}%)")